# Simple Campus Menu Agent

This notebook creates a simple agent that can answer questions about today's campus menu.
It uses the `get_menu` tool from `getmenus.py` to look up data from the database.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from IPython.display import display, Markdown
from loguru import logger

# Import the get_menu tool and helper functions from getmenus.py
from src.getmenus import get_menu, get_today, setup_logging

setup_logging()
load_dotenv()
print("Environment loaded.")

In [ ]:
logger.info(f"Today's date (Seoul time): {get_today()}")

## Create the Language Model

Connect to Azure OpenAI using credentials stored in `.env`.

In [ ]:
# Create the language model using Azure OpenAI credentials from the .env file
llm = ChatOpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    timeout=300,
    temperature=0.5,
    max_tokens=25000
)

logger.info("LLM ready.")


## Build the Agent

Create the agent with the `get_menu` tool and a helpful system prompt.

In [ ]:
today = get_today()

# The system prompt tells the agent how to behave and what today's date is
SYSTEM_PROMPT = (
    f"You are a helpful campus menu assistant. Today's date is {today} (Asia/Seoul). "
    "Use the get_menu tool to fetch menu data for the requested date. "
    "Translate all Korean names (universities, restaurants, dishes) to English. "
    "Group the answer by university, then by restaurant. "
    "Include meal type, dish names, serving time, and price when available."
)

# Build the agent with the get_menu tool attached
agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_menu]
)

logger.info("Agent ready.")

## Ask the Agent a Question

Change the `question` variable below and run the cell to get an answer.

In [ ]:
# Change this question to ask anything about the menu
question = "What is on the menu today?"

In [ ]:
logger.info("Question: {!r}", question)

# Run the agent with our question
result = agent.invoke({"messages": [{"role": "user", "content": question}]})

# Extract the final text reply from the agent's response messages
answer = ""
for message in reversed(result.get("messages", [])):
    content = getattr(message, "content", None)
    if isinstance(content, str) and content.strip():
        answer = content
        break

logger.info("Answer: {} chars", len(answer))

# Display the answer nicely formatted in the notebook
display(Markdown(answer))

## Email-sending


In [ ]:
import httpx
import re
from typing import Optional

# Ensure logs directory exists (setup_logging() from getmenus.py likely configures loguru handlers,
# but we still make sure the folder is present for any file handlers)
os.makedirs("logs", exist_ok=True)


def send_agent_response_via_smtp2go(html_body: str, subject: Optional[str] = None) -> dict | None:
    """
    Send the agent's HTML response via SMTP2GO using the /v3/email/send endpoint.
    Reads SMTP2GO_API_KEY, SMTP2GO_SENDER_EMAIL, SMTP2GO_RECIPIENT_EMAIL from environment.
    Logs each step with loguru.
    Returns the parsed JSON response on success, or None on error.
    """
    logger.info("Preparing to send email via SMTP2GO")

    api_key = os.environ.get("SMTP2GO_API_KEY")
    sender = os.environ.get("SMTP2GO_SENDER_EMAIL")
    recipient = os.environ.get("SMTP2GO_RECIPIENT_EMAIL")

    if not api_key or not sender or not recipient:
        logger.error("Missing SMTP2GO configuration in environment: SMTP2GO_API_KEY/SENDER/RECIPIENT")
        return None

    if not subject:
        subject = "Campus Menu Agent Response"

    # Create a simple plain-text fallback by stripping basic HTML tags
    text_body = re.sub(r"<[^>]+>", "", html_body).strip()
    if len(text_body) == 0:
        text_body = "See HTML body."

    payload = {
        "sender": sender,
        "to": [recipient],
        "subject": subject,
        "html_body": html_body,
        "text_body": text_body
    }

    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json",
        "X-Smtp2go-Api-Key": api_key
    }

    url = "https://api.smtp2go.com/v3/email/send"
    logger.info("Sending email to {recipient} with subject '{subject}'", recipient=recipient, subject=subject)
    logger.debug("SMTP2GO payload keys: {}", list(payload.keys()))

    try:
        with httpx.Client(timeout=30.0) as client:
            resp = client.post(url, json=payload, headers=headers)
        logger.info("SMTP2GO response status: {}", resp.status_code)

        try:
            resp_json = resp.json()
        except ValueError:
            logger.error("Failed to decode JSON from SMTP2GO response: {}", resp.text)
            return None

        # Log response details (keep it concise)
        logger.debug("SMTP2GO response JSON: {}", resp_json)

        if resp.status_code == 200:
            succeeded = resp_json.get("data", {}).get("succeeded")
            logger.info("Email send result: succeeded={}, failed={}",
                        succeeded, resp_json.get("data", {}).get("failed"))
            return resp_json
        else:
            logger.error("SMTP2GO returned error: {}", resp_json)
            return resp_json
    except Exception as e:
        logger.exception("Exception while sending email via SMTP2GO: {}", str(e))
        return None